In [ ]:
# =============================================================================
# Minimal main loop (UTC+0) - per-cycle output with leading separator (simplified)
# - No exception handling; keep only the essential steps.
# - Each cycle: clear screen, print separator, print two summary lines, call sync.
# - No accumulation of logs across cycles. No ms printed in per-cycle summary.
# =============================================================================
from typing import Optional
import time
import sqlite3
import os
import sys

# =============================================================================
# Local imports
# =============================================================================
import utils as utils
from database_codes.bch_usdt_1m import sync_bchusdt_1m_from_ms

# =============================================================================
# Configuration / constants
# =============================================================================
POLL_SECONDS = 10
SEPARATOR    = "-" * 60

# =============================================================================
# Helper: get max value from a table column
# =============================================================================
def _sql_max(conn: sqlite3.Connection, table: str, column: str) -> Optional[object]:
    cur = conn.cursor()
    cur.execute(f'SELECT MAX("{column}") FROM "{table}"')
    row = cur.fetchone()
    return row[0] if row and row[0] is not None else None

# =============================================================================
# Helper: clear the terminal screen
# =============================================================================
def _clear_screen() -> None:
    if os.name == "nt":
        os.system("cls")
    else:
        sys.stdout.write("\033[2J\033[H")
        sys.stdout.flush()

# =============================================================================
# Print cycle header: clear screen and print leading separator
# =============================================================================
def _print_cycle_header() -> None:
    _clear_screen()
    print(SEPARATOR)

# =============================================================================
# Print the per-cycle lines (timestamped summary)
# =============================================================================
def _print_cycle_lines(base_table: str, base_max_display: str) -> None:
    ts = utils.now_utc_str()
    print(f"[{ts}] Base max open_time: {base_max_display}")
    print(f"[{ts}] Sync started for base table '{base_table}'")

# =============================================================================
# Main loop (very simple)
# - Loads configuration and validates required keys (database.db_path, database.dev_data.table_name).
# - Each cycle:
#     * queries the base table's MAX(open_time) and MAX(open_time_ms),
#     * clears the screen and prints a leading separator and two summary lines,
#     * computes start_ms for the sync,
#     * calls sync_bchusdt_1m_from_ms(start_ms),
#     * sleeps POLL_SECONDS seconds.
# - No exception handling is performed; terminate with Ctrl+C or the usual OS signal.
# =============================================================================
def main_loop() -> None:
    # Load configuration using the helper in utils
    cfg     = utils._load_config()

    # Extract database path; exit if missing or not a string
    db_path = cfg.get("database", {}).get("db_path")
    if not db_path or not isinstance(db_path, str):
        print("Database path not configured (database.db_path). Exiting.")
        return

    # Extract the base table name; exit if missing
    base_table = cfg.get("database", {}).get("dev_data", {}).get("table_name")
    if not base_table:
        print("Base table name not configured (database.dev_data.table_name). Exiting.")
        return

    # Initial screen clear and informational header
    _clear_screen()
    print("Minimal main (UTC+0) - per-cycle output (newest only)\n")
    print(SEPARATOR)

    # Simple infinite polling loop
    while True:
        # Open sqlite connection and fetch the two max values
        with sqlite3.connect(db_path) as conn:
            base_max_time    = _sql_max(conn, base_table, "open_time")
            base_max_time_ms = _sql_max(conn, base_table, "open_time_ms")

        # Decide how to display the base max: prefer human-readable open_time,
        # fall back to converting ms, otherwise show "<no data>"
        if base_max_time is not None:
            base_max_display = str(base_max_time)[:24]
        elif base_max_time_ms is not None:
            base_max_display = utils.ms_to_utc_str(int(base_max_time_ms))
        else:
            base_max_display = "<no data>"

        # Print cycle header and summary lines
        _print_cycle_header()
        _print_cycle_lines(base_table, base_max_display)

        # Determine start_ms for synchronization: next ms after base_max_time_ms or now
        if base_max_time_ms is not None:
            start_ms = int(base_max_time_ms) + 1
        else:
            start_ms = utils.now_utc_ms()

        # Call the sync function (it may print its own details)
        sync_bchusdt_1m_from_ms(start_ms)

        # Sleep until next cycle
        time.sleep(POLL_SECONDS)

# =============================================================================
# Entrypoint
# =============================================================================
if __name__ == "__main__":
    main_loop()

Minimal main (UTC+0) - per-cycle output (newest only)

------------------------------------------------------------
------------------------------------------------------------
[2025-10-31 10:08:36] Base max open_time: 2025-10-31 10:05:00
[2025-10-31 10:08:36] Sync started for base table 'bchusdt_1m'
Appended 3 rows into 'bchusdt_1m'
------------------------------------------------------------
[2025-10-31 10:08:47] Base max open_time: 2025-10-31 10:08:00
[2025-10-31 10:08:47] Sync started for base table 'bchusdt_1m'
No klines returned from Binance for the requested start.
------------------------------------------------------------
[2025-10-31 10:08:58] Base max open_time: 2025-10-31 10:08:00
[2025-10-31 10:08:58] Sync started for base table 'bchusdt_1m'
No klines returned from Binance for the requested start.
------------------------------------------------------------
[2025-10-31 10:09:08] Base max open_time: 2025-10-31 10:08:00
[2025-10-31 10:09:08] Sync started for base table 'bchu